### Parking Meter Data

### 0. Summary of the dataset

#### 0.1 What the raw table contains
The original parking meter dataset is a wide table combining:
- **Geometry / inventory fields:** x, y, location
- **Zone identifier:** node
- **Pricing fields:** rate, rate2, ratetimea, ratetimeb
- **Operating hours fields:** date1, date2, time1, time2
- **Restriction/rest fields:** restdates, resttime1, resttime2
- **Duration limit:** maxhours  
(See **2.1** for detailed column behavior and value patterns.)

#### 0.2 Key data characteristics
- **Mixed observational units:** the table simultaneously contains meter/point-level info (coordinates), zone-level info (node), and rule-level info (rates/hours).  
- **location is not a unique key:** the same location label can appear many times and may map to multiple node values (street labels spanning zones, or multiple meters under one label).  
(See **2.2** for duplication and inconsistency patterns.)

#### 0.3 Why a single “one row per (node, location)” table is problematic
- For the same (node, location), other columns can legitimately differ (different meters/segments), so collapsing requires arbitrary “pick-one” rules.
- Rate and schedule fields create one-to-many relationships, which would produce repeated rows (cartesian products) if kept in one table.  
(See **2.2** for the implications.)

#### 0.4 Current restructuring approach (normalized design)
To remove ambiguity and make joins easier, we split the raw table into four datasets:
- **Parking_zone:** one row per zone (zone_id = node), includes meter counts and location lists (**3.1.1**).
- **Parking_meters:** one row per meter (proxy id using rounded coords + zone), keeps x, y, location (**3.1.1**).
- **Parking_rates:** one row per meter's rate (A/B windows) per zone (**3.1.2**).
- **Parking_hours:** one row per meter's schedule rule per zone (ACTIVE vs REST) (**3.1.2**).

This normalized structure preserves all rule variations while keeping the primary join key (zone_id) consistent with the transaction dataset.

### 1. Dataset Overview

Import

In [2]:
import polars as pl

parking_meter_info = pl.read_csv("raw_data/Parking Meter Information.csv")
parking_meter_info.head(3)

_id,x,y,objectid,terminal_id,node,location,lot,rate,maxhours,date1,time1,date2,time2,restdates,resttime1,resttime2,specialev,created_user,created_date,last_edited_user,last_edited_date,ratetimea,ratetimeb,rate2
i64,f64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
103060,-79.922827,40.46249,225,"""301520-SHARVD0001""","""SHER-HAR-L""","""SHERIDAN HARVARD LOT""","""Lot""","""$1 Per Hour""","""4 Hours""","""Mon - Sat""","""~ 8AM - 10PM""",""" """,""" """,""" """,""" """,""" ""","""No""",null,null,null,null,""" """,""" """,""" """
103061,-79.92353,40.462718,226,"""301521-SHARVD0002""","""SHER-HAR-L""","""SHERIDAN HARVARD LOT""","""Lot""","""$1 Per Hour""","""4 Hours""","""Mon - Sat""","""~ 8AM - 10PM""",""" """,""" """,""" """,""" """,""" ""","""No""",null,null,null,null,""" """,""" """,""" """
103062,-79.923127,40.461796,227,"""302521-KIRKWD0001""","""SHER-KIR-L""","""SHERIDAN KIRKWOOD LOT""","""Lot""","""$1 Per Hour""","""4 Hours""","""Mon - Sat""","""~ 8AM - 10PM""",""" """,""" """,""" """,""" """,""" ""","""No""",null,null,null,null,""" """,""" """,""" """


**Non fields level documentation found for this dataset**

Anyways, we can reasonably believe:   
Location / geometry: x, y (coordinates)   
Identifiers: objectid, terminal_id, node  
Where it is described: location, lot  
Pricing & limits: rate, maxhours  
Rules windows: date/time fields like date1, time1, date2, time2, plus fields for restrictions / special events (e.g., restdates, resttime1, resttime2, specialev)  
Record metadata: created/edited user and timestamps  

In [3]:
# Replace empty strings with null
parking_meter_info = pl.read_csv("raw_data/Parking Meter Information.csv", null_values=[" "])
parking_meter_info.head(1)

_id,x,y,objectid,terminal_id,node,location,lot,rate,maxhours,date1,time1,date2,time2,restdates,resttime1,resttime2,specialev,created_user,created_date,last_edited_user,last_edited_date,ratetimea,ratetimeb,rate2
i64,f64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
103060,-79.922827,40.46249,225,"""301520-SHARVD0001""","""SHER-HAR-L""","""SHERIDAN HARVARD LOT""","""Lot""","""$1 Per Hour""","""4 Hours""","""Mon - Sat""","""~ 8AM - 10PM""",null,null,null,null,null,"""No""",null,null,null,null,null,null,null


**Data summary**

Many columns have a huge percentage of null values (we will see pivoting could fix it in **section 3**)

In [4]:
parking_meter_info.describe()

statistic,_id,x,y,objectid,terminal_id,node,location,lot,rate,maxhours,date1,time1,date2,time2,restdates,resttime1,resttime2,specialev,created_user,created_date,last_edited_user,last_edited_date,ratetimea,ratetimeb,rate2
str,f64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""",1091.0,1026.0,1026.0,1091.0,"""1091""","""1091""","""1091""","""59""","""1022""","""1020""","""1020""","""1020""","""108""","""108""","""92""","""92""","""64""","""1020""","""0""","""0""","""0""","""0""","""10""","""10""","""10"""
"""null_count""",0.0,65.0,65.0,0.0,"""0""","""0""","""0""","""1032""","""69""","""71""","""71""","""71""","""983""","""983""","""999""","""999""","""1027""","""71""","""1091""","""1091""","""1091""","""1091""","""1081""","""1081""","""1081"""
"""mean""",103605.0,-79.970071,40.443586,486.528873,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",315.088876,0.030602,0.015408,317.007915,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""",103060.0,-80.041461,40.383086,0.0,"""301520-SHARVD0001""","""18-CARSO-L""","""10TH ST""","""LOT""","""$1 Per Hour""","""10 Hours""","""Mon - Sat""","""~ 8AM - 10PM""","""Fri - Sat""","""~ 8AM - 10PM""","""Fri - Sat""","""10PM - 2AM""","""3PM - 6PM""","""No""",null,null,null,null,"""~ 8AM - 2PM""","""~ 2PM - 6PM""","""$2.50 Per Hour"""
"""25%""",103333.0,-79.997908,40.437817,208.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",103605.0,-79.967356,40.44395,486.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",103878.0,-79.946664,40.453321,760.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""",104150.0,-79.896882,40.49001,1040.0,"""PBP427""","""Z - Inactive/Removed Terminals""","""WYLIE AVE""","""Lot""","""Dynamic""","""No Max""","""Mon - Thurs""","""~ 8AM - 6PM""","""Fri - Sat""","""~ 8AM - 12AM""","""Mon - Fri""","""7AM - 9AM""","""4PM - 6PM""","""Yes""",null,null,null,null,"""~ 8AM - 2PM""","""~ 2PM - 6PM""","""$2.50 Per Hour"""


**Check unique value of some columns**

In [5]:
# Check fields unique values
print(f"Unique values in lot: {parking_meter_info.select('lot').unique().to_series().to_list()}")
print(f"Unique values in specialev: {parking_meter_info.select('specialev').unique().to_series().to_list()}")
print(f"Unique values in node: {parking_meter_info.select('node').unique().to_series().to_list()[:7]}")

Unique values in lot: ['Lot', 'LOT', None]
Unique values in specialev: [None, 'Yes', 'No']
Unique values in node: ['NORTHSHORE', 'BROOKLINE', 'OAKLAND4', '18-CARSO-L', 'W CIRC DR', 'DOWNTOWN2', 'STRIPDIST']


In [6]:
# Exclude column lot, specialev and ids which are not really informative
parking_cols = ["x", "y", "node", "location"]

### 2. Dataset Analysis

#### 2.1. Single columns meaning analysis

##### 2.1.1. Date & time fields 
**Let's check why there are two time in dataset (time1, time2, date1, date2)**
From results bellow we conclude:
* date1 - time1 are weekdays active hours
* date2 - time2 are weekedns active hours

In [7]:
# Do value appear in time1 and time2 at same time?
time_tmp = parking_meter_info.filter(
    pl.col("time1").is_not_null()
    & pl.col("time2").is_not_null()
    )
print(f"Number of rows with values both in time1 and time2: {len(time_tmp)}")
time_tmp.select("time1", "time2").head(2)

# Confirmed, time1 and time2 always have values at the same time
# Because the number of rows that have in time2 in full dataset = 108

Number of rows with values both in time1 and time2: 108


time1,time2
str,str
"""~ 8AM - 10PM""","""~ 8AM - 10PM"""
"""~ 8AM - 10PM""","""~ 8AM - 10PM"""


In [8]:
# Do time1 and time2 have same values?
time_values_tmp = parking_meter_info.filter(
    pl.col("time1").is_not_null()
    & pl.col("time2").is_not_null()
    & (pl.col("time1") != pl.col("time2"))
    )
print(f"Number of rows with different values in time1 and time2: {len(time_values_tmp)}")
print(f"First few rows with different values in time1 and time2:")
print(time_values_tmp.select("time1", "time2").unique())

# 99/108 has different values

Number of rows with different values in time1 and time2: 99
First few rows with different values in time1 and time2:
shape: (2, 2)
┌──────────────┬──────────────┐
│ time1        ┆ time2        │
│ ---          ┆ ---          │
│ str          ┆ str          │
╞══════════════╪══════════════╡
│ ~ 8AM - 10PM ┆ ~ 8AM - 12AM │
│ ~ 8AM - 6PM  ┆ ~ 8AM - 12AM │
└──────────────┴──────────────┘


In [9]:
# Check relationship between time1 - date1, time2 - date2
time_date_tmp = parking_meter_info.filter(
    pl.col("time1").is_not_null()
    & pl.col("date1").is_not_null()
    )
print(f"Number of rows with values in time1 and date1: {len(time_date_tmp)}")
print(f"First few rows with values in time1 and date1:")
print(time_date_tmp.select("time1", "date1").head(2))

time_date_tmp2 = parking_meter_info.filter(
    pl.col("time2").is_not_null()
    & pl.col("date2").is_not_null()
    )
print(f"Number of rows with values in time2 and date2: {len(time_date_tmp2)}")
print(f"First few rows with values in time2 and date2:")
print(time_date_tmp2.select("time2", "date2").head(2))

# Confirmed, time1 and date1 always have values at the same time, time2
# and date2 also always have values at the same time

Number of rows with values in time1 and date1: 1020
First few rows with values in time1 and date1:
shape: (2, 2)
┌──────────────┬───────────┐
│ time1        ┆ date1     │
│ ---          ┆ ---       │
│ str          ┆ str       │
╞══════════════╪═══════════╡
│ ~ 8AM - 10PM ┆ Mon - Sat │
│ ~ 8AM - 10PM ┆ Mon - Sat │
└──────────────┴───────────┘
Number of rows with values in time2 and date2: 108
First few rows with values in time2 and date2:
shape: (2, 2)
┌──────────────┬───────────┐
│ time2        ┆ date2     │
│ ---          ┆ ---       │
│ str          ┆ str       │
╞══════════════╪═══════════╡
│ ~ 8AM - 10PM ┆ Fri - Sat │
│ ~ 8AM - 10PM ┆ Fri - Sat │
└──────────────┴───────────┘


In [10]:
# Check relationship between specialev and time1, time2
specialev_time_tmp = parking_meter_info.filter(
    (pl.col("specialev") == "No")
    & pl.col("time1").is_not_null()
    & pl.col("time2").is_not_null()
    )
print(f"Number of rows with values in specialev, time1 and time2: {len(specialev_time_tmp)}")
print(f"First few rows with values in specialev, time1 and time2:")
print(specialev_time_tmp.select("specialev", "time1", "time2").head(2))

# time1 and time2 have values only when specialev is "No"
# which makes sense since specialev indicates whether the parking meter is for special events,
# and time1 and time2 indicate the time restrictions for parking.

Number of rows with values in specialev, time1 and time2: 108
First few rows with values in specialev, time1 and time2:
shape: (2, 3)
┌───────────┬──────────────┬──────────────┐
│ specialev ┆ time1        ┆ time2        │
│ ---       ┆ ---          ┆ ---          │
│ str       ┆ str          ┆ str          │
╞═══════════╪══════════════╪══════════════╡
│ No        ┆ ~ 8AM - 10PM ┆ ~ 8AM - 10PM │
│ No        ┆ ~ 8AM - 10PM ┆ ~ 8AM - 10PM │
└───────────┴──────────────┴──────────────┘


**Are columns resttime1 and restime2 related with restdates?**

The following code shows resttime1 and resttime2 are describing info of restdate.

In [11]:
# Check for each row that has value in restdates whether it also has values in resttime1 and resttime2
rest_dates_tmp = parking_meter_info.filter(
    pl.col("restdates").is_not_null()
    & (pl.col("resttime1").is_not_null() | pl.col("resttime2").is_not_null())
    )
print(f"Number of rows with values in restdates, resttime1 and resttime2: {len(rest_dates_tmp)}")
print(f"First few rows with values in restdates, resttime1 and resttime2:")
print(rest_dates_tmp.select("restdates", "resttime1", "resttime2").head(2))

# Confirmed

Number of rows with values in restdates, resttime1 and resttime2: 92
First few rows with values in restdates, resttime1 and resttime2:
shape: (2, 3)
┌───────────┬───────────┬───────────┐
│ restdates ┆ resttime1 ┆ resttime2 │
│ ---       ┆ ---       ┆ ---       │
│ str       ┆ str       ┆ str       │
╞═══════════╪═══════════╪═══════════╡
│ Mon - Fri ┆ 7AM - 9AM ┆ 4PM - 6PM │
│ Mon - Fri ┆ 7AM - 9AM ┆ 4PM - 6PM │
└───────────┴───────────┴───────────┘


add to parking cols

In [12]:
parking_cols = parking_cols + ["date1", "date2", "time1", "time2", "restdates", \
                "resttime1", "resttime2"]

##### 2.1.2 Rate columns
**What do rate and rate 2 represent?**

The following code shows ratetimea, ratetimeb are correlated with rate and rate1
* rate rapresent rate for retetimea
* rate2 is for ratetimeb
* for rows that there aren't value in rate2, rate is for all time
* rate2, ratetimea/b appear at same time (no value in rate2 -> no value in ratetimea/b)


An interesting point here is that, in the date & time analysis above, we have time1 and time2, and now we have rate (not rate1) and rate2 instead.

The reason is that, different from the previous case where time1 always has no-null value when date1 has (1080 rows for both columns). ratetimea has no-null value only when ratetimeb and rate2 have. So, most of the time we use rate column (which is for all time), and only when there are values in rate2, ratetimea/b we use rate for ratetimea and rate2 for ratetimeb. 

This logic is quite confusing, a systematic arrangement should be done for these columns.

In [13]:
# Check whether value of ratetimea, ratetimeb appears at same time
len(parking_meter_info.filter(
    pl.col("ratetimea").is_not_null()
    & pl.col("ratetimeb").is_null()
    ))
# Confirmed

0

In [14]:
# Check wether existance of values in rate2 are related to rate, ratetimea and ratetimeb
rate_time_tmp = parking_meter_info.filter(    
    pl.col("rate").is_not_null()
    & pl.col("rate2").is_not_null()
    & pl.col("ratetimea").is_not_null()
    & pl.col("ratetimeb").is_not_null()
    )
print(f"Number of rows with values in rate2, ratetimea and ratetimeb: {len(rate_time_tmp)}")
print(f"First few rows with values in rate2, ratetimea and ratetimeb:")
print(rate_time_tmp.select("rate2", "ratetimea", "ratetimeb").head(2))

# Confirmed, because the number of rows are 10 which corresponds to the number of
# rows in full datasets for ratetimea/b and rate 2

Number of rows with values in rate2, ratetimea and ratetimeb: 10
First few rows with values in rate2, ratetimea and ratetimeb:
shape: (2, 3)
┌────────────────┬─────────────┬─────────────┐
│ rate2          ┆ ratetimea   ┆ ratetimeb   │
│ ---            ┆ ---         ┆ ---         │
│ str            ┆ str         ┆ str         │
╞════════════════╪═════════════╪═════════════╡
│ $2.50 Per Hour ┆ ~ 8AM - 2PM ┆ ~ 2PM - 6PM │
│ $2.50 Per Hour ┆ ~ 8AM - 2PM ┆ ~ 2PM - 6PM │
└────────────────┴─────────────┴─────────────┘


**Let's check unique values in rate and rate2**

They are all per hour price (exept null and dynamic)

In [15]:
print(f"Unique values of rate column: {parking_meter_info.select("rate").unique().to_series().to_list()}")

Unique values of rate column: ['$4 Per Hour', '$2 Per Hour', None, 'Dynamic', '$1.00 Per Hour', '$3 Per Hour', '$1.50 Per Hour', '$2.00 Per Hour', '$3.00 Per Hour', '$1 Per Hour']


In [16]:
print(f"Unique values of rate2 column: {parking_meter_info.select("rate2").unique().to_series().to_list()}")

Unique values of rate2 column: ['$2.50 Per Hour', None]


In [17]:
parking_meter_info = parking_meter_info.with_columns([
    # Let's simplify it, so we consider dynamic as None
    # (pl.col("rate").str.to_lowercase() == "dynamic")
    #   .fill_null(False)
    #   .alias("rate_is_dynamic"),
    # numeric parse for non-dynamic: "$1.50 Per Hour" -> 1.50
    pl.when(
        pl.col("rate").is_not_null()
        & (pl.col("rate").str.to_lowercase() != "dynamic")
    )
    .then(
        pl.col("rate")
          .str.replace_all(r"[^0-9.]", "")
          .replace("", None)
          .cast(pl.Float64)
    )
    .otherwise(None)
    .alias("rate_hours"),

    pl.when(pl.col("rate2").is_not_null())
    .then(
        pl.col("rate2")
          .str.replace_all(r"[^0-9.]", "")
          .replace("", None)
          .cast(pl.Float64)
    )
    .otherwise(None)
    .alias("rate2_hours"),
])
parking_meter_info.select(["rate", "rate2", "rate_hours","rate2_hours"]).head(3)

rate,rate2,rate_hours,rate2_hours
str,str,f64,f64
"""$1 Per Hour""",null,1.0,null
"""$1 Per Hour""",null,1.0,null
"""$1 Per Hour""",null,1.0,null


add to parking_cols list

In [18]:
parking_cols = parking_cols + ["rate", "rate2","rate_hours", "rate2_hours", \
                "ratetimea", "ratetimeb"]

##### 2.1.3 Maxhours

Following code shows that all values of maxhours columns are either no max or hours

In [19]:
# Check whether hours are used for all values
maxhour_check = parking_meter_info.filter(
    pl.col("maxhours").str.to_lowercase().str.contains("hours")
)
maxhour_check.height
# Not confirmed, only 565/1080 have hours in it

565

In [20]:
# Let's check what other values it contains
maxhour_check2 = parking_meter_info.filter(
    pl.col("maxhours").str.to_lowercase().str.contains("hours") == False
)
maxhour_check2.select("maxhours").unique()

maxhours
str
"""No Max"""


In [21]:
parking_meter_info = parking_meter_info.with_columns([
    # numeric hours if present (extract first number)
    pl.when(
        pl.col("maxhours").is_not_null()
        # Check if field contains digit
        & pl.col("maxhours").str.to_lowercase().str.contains(r"\d")
    )
    .then(
        pl.col("maxhours")
          .str.extract(r"(\d+(\.\d+)?)", 1)   # "4", "1.5"
          .cast(pl.Float64)
    )
    .otherwise(None)
    .alias("maxhours_hours"),
])

parking_cols.append("maxhours_hours")

**Filtered dataset for parking**

There are a lot null values still -> we need pivoting!

In [22]:
parking_meter_info_filtered = parking_meter_info.select(parking_cols)

#### 2.2. Strange pattern analysis

##### 2.2.1 Location rows
**Many location have multiple rows**

The reason we believed is that they represent different meters in that location 

We have variable in rows! -> Later we should add a column called Number of meters

In [23]:
# Check unique values in parking_meter_info_filtered.location
unique_locations = parking_meter_info_filtered.select("location").sort(by="location").unique().to_series().to_list()
print("Number of rows in s:", parking_meter_info_filtered.height)
print("Number of unique locations in s.location:", len(unique_locations))
print(unique_locations)

Number of rows in s: 1091
Number of unique locations in s.location: 234
['10TH ST', '12TH ST', '13TH ST', '14TH ST', '17TH ST', '18TH & CARSON LOT', '18TH & SIDNEY LOT', '18TH ST', '19TH & CARSON LOT', '19TH ST', '1ST AVE', '20TH & SIDNEY LOT', '20TH ST', '21ST ST', '22ND ST', '24TH ST', '2ND AVE', '3RD AVE', '42ND & BUTLER LOT', '4TH AVE', '52ND & BUTLER LOT', '5TH AVE', '5TH AVE.', '6TH AVE', '7TH AVE', '7TH ST', '9TH ST', 'ALLEGHENY SQ', 'ANSLEY/BEATTY LOT', 'ARCH ST', 'ART ROONEY DR', 'ASTEROID WARRINGTON LOT', 'ATWOOD ST', 'BAKERY SQ BLVD', 'BARLETT AVE', 'BAUM BLVD', 'BAYARD ST', 'BEACON AVE', 'BEACON BARTLETT LOT', 'BEDFORD AVE', 'BEDFORD SQ', 'BEECHVIEW AVE', 'BEECHVIEW LOT', 'BELLEFONTE ST', 'BIGELOW BLVD', 'BLVD OF ALLIES', 'BOYD ST', 'BRIGHTON RD', 'BROAD ST', 'BROADWAY AVE', 'BROOKLINE BLVD', 'BROOKLINE LOT', 'BROWNSVILLE RD', 'BROWNSVILLE SANKEY LOT', 'BUTLER ST', 'CALIFORNIA AVE', 'CEDAR AVE', 'CEDARVILLE ST', 'CENTRE AVE', 'CHATHAM SQ', 'CHILDRENS WAY', 'COLLEGE AVE', 'C

**Do rows that have same location also have same node?**

Answer: No, some don't

The reason that we believed is that: location represent a street name (or a generic label like “Virtual terminal…”). A single street can belong to multiple zones, and nodes represent zones, so the same location can legitimately appear with different nodes.

For example, long streets like FORBES AVE, PENN AVE, CENTRE AVE run through multiple neighborhoods.

The values in x_span and y_span columns of each location group confirms that some rows with same location rows could be physically far apart along the street.

Another explanation is: the location sits right on a zone boundary (data assignment inconsistency could be done to further confirms it)

Note:
* One degree of latitude equals approximately 364,000 feet (69 miles), one minute equals 6,068 feet (1.15 miles), and one-second equals 101 feet.    
* One-degree of longitude equals 288,200 feet (54.6 miles), one minute equals 4,800 feet (0.91 mile), and one second equals 80 feet.

Info from "How much distance does a degree, minute, and second cover on your maps?" (url:https://www.usgs.gov/faqs/how-much-distance-does-a-degree-minute-and-second-cover-your-maps)

In [24]:
same_node_cols = (
    parking_meter_info_filtered
    .group_by("location")
    .agg([
        (pl.col("node").n_unique() <= 1).alias("is_node_same"),
        pl.col("node").unique().alias("unique_nodes"),
        pl.col("x").min().alias("x_min"),
        pl.col("x").max().alias("x_max"),
        pl.col("y").min().alias("y_min"),
        pl.col("y").max().alias("y_max"),
        pl.len().alias("n_rows"),
    ]
    )
    .with_columns([
        (pl.col("x_max") - pl.col("x_min")).alias("x_span"),
        (pl.col("y_max") - pl.col("y_min")).alias("y_span"),
    ])
    .sort(["x_span","y_span"], descending=True)
)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(2000)
(
    same_node_cols
    .filter(pl.col("is_node_same")==False)
    .select(["location", "is_node_same", "unique_nodes", "x_span", "y_span"])
)

location,is_node_same,unique_nodes,x_span,y_span
str,bool,list[str],f64,f64
"""Virtual terminal for ParkMobil…",false,"[""WALT-WAR-L"", ""KNOXVILLE""]",null,null
"""Virtual Terminal for ParkMobil…",false,"[""18-CARSO-L"", ""20-SIDNE-L"", … ""HILL-DIST""]",null,null
"""FORBES AVE""",false,"[""DOWNTOWN1"", ""UPTOWN2"", … ""SQ.HILL1""]",0.083886,0.005892
"""PENN AVE""",false,"[""DOWNTOWN1"", ""DOWNTOWN2"", … ""EASTLIB""]",0.083055,0.024955
"""CENTRE AVE""",false,"[""HILL-DIST-2"", ""UPTOWN1"", … ""HILL-DIST""]",0.067421,0.021456
"""LIBERTY AVE""",false,"[""DOWNTOWN1"", ""DOWNTOWN2"", ""BLOOMFIELD""]",0.065389,0.021998
"""BAUM BLVD""",false,"[""BLOOMFIELD"", ""EASTLIB""]",0.021758,0.006127
"""BROWNSVILLE RD""",false,"[""CARRICK"", ""KNOXVILLE""]",0.013086,0.031007
"""FIFTH AVE""",false,"[""OAKLAND2"", ""OAKLAND3""]",0.008513,0.004191


#### 2.2.2. Frequent void values in columns

As we’ve seen, several columns in this table are not true variables: they store multiple pieces of information by splitting one concept across “paired” columns.

There are date1/time1, date2/time2, restdates; rate2, reatimea/b.

To make the data easier to analyze, we should **pivot** these paired columns into a long format.

In [25]:
parking_meter_info_filtered.head(5)

x,y,node,location,date1,date2,time1,time2,restdates,resttime1,resttime2,rate,rate2,rate_hours,rate2_hours,ratetimea,ratetimeb,maxhours_hours
f64,f64,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,f64
-79.922827,40.46249,"""SHER-HAR-L""","""SHERIDAN HARVARD LOT""","""Mon - Sat""",null,"""~ 8AM - 10PM""",null,null,null,null,"""$1 Per Hour""",null,1.0,null,null,null,4.0
-79.92353,40.462718,"""SHER-HAR-L""","""SHERIDAN HARVARD LOT""","""Mon - Sat""",null,"""~ 8AM - 10PM""",null,null,null,null,"""$1 Per Hour""",null,1.0,null,null,null,4.0
-79.923127,40.461796,"""SHER-KIR-L""","""SHERIDAN KIRKWOOD LOT""","""Mon - Sat""",null,"""~ 8AM - 10PM""",null,null,null,null,"""$1 Per Hour""",null,1.0,null,null,null,4.0
-79.922689,40.461792,"""SHER-KIR-L""","""SHERIDAN KIRKWOOD LOT""","""Mon - Sat""",null,"""~ 8AM - 10PM""",null,null,null,null,"""$1 Per Hour""",null,1.0,null,null,null,4.0
-79.922329,40.461512,"""SHER-KIR-L""","""SHERIDAN KIRKWOOD LOT""","""Mon - Sat""",null,"""~ 8AM - 10PM""",null,null,null,null,"""$1 Per Hour""",null,1.0,null,null,null,4.0


### 3. Tidy Data

As we discussed before, the current columns are not really ideal (some variables are not include, and some variables are splitted into multiple columns)

We could create these new variables (columns):
* number_meters -> remove duplicated node/location rows
* schedule_type -> active_time, rest_time
* day_range -> e.g. "Mon–Sat", "Mon–Fri", "Weekend" (remove date1, date2)
* time_range -> e.g. "8AM–10PM", "7AM–9AM" (remove time1, time2)
* rate_window -> a or b (tracks ratetimea/rate vs ratetimeb/rate2)
* rate_time -> e.g. "8AM–2PM" (from ratetimea / ratetimeb)
* rate -> the price (from rate / rate2)
Columns removed: ["rate", "rate2", "ratetimea", "ratetimeb", "date1", "date2", "time1", "time2", "restdates", "resttime1", "resttime2"]

However, the datasets contains multiple types of observational units:
* Location/Zone
* Rates
* Active / resting hours
* Number of meters

Which leads to two issues:

**Issue 1**: One node/location produces repeated rows   
Which is not an error but it’s telling us that we have one-to-many relationships:    
* one zone → many rate windows
* one zone → many hours schedules (paid hours vs restricted/rest)
* one zone → many meters

**Issue 2**: Same node/location has different values.   
For records that share the same (node or location), other fields (e.g., rate, hours, max time, restrictions) can still differ because they represent different meters/blocks within that same street/zone.
So if we try to collapse everything into one row per (node or location), we run into an ambiguity: which set of values should represent the group

Thus, Including all these info in one single dataframe would be unclear, and makes joining other datasets harder.

In addition, many fields could be converted to more appropriate type (e.g. $1 Per Hour -> 1)

**Let's try to address these issues by break it down to 4 datasets: Parking_zone, Parking_Rates, Parking_hours and Parking_meters**

**Code that shows issue 1**

In [26]:
id_cols = ["x", "y", "node", "location","maxhours_hours"]  # add/remove as needed

# Schedule type
# for date, time 1
active_1 = (
    parking_meter_info_filtered.select(id_cols + ["date1", "time1"])
      .rename({"date1": "day_range", "time1": "time_range"})
      .with_columns([
          # create schedule_type column
          pl.lit("active_time").alias("schedule_type"),
      ])
)

active_2 = (
    parking_meter_info_filtered.select(id_cols + ["date2", "time2"])
      .rename({"date2": "day_range", "time2": "time_range"})
      .with_columns([
          pl.lit("active_time").alias("schedule_type"),
      ])
)

rest_1 = (
    parking_meter_info_filtered.select(id_cols + ["restdates", "resttime1"])
      .rename({"restdates": "day_range", "resttime1": "time_range"})
      .with_columns([
          pl.lit("rest_time").alias("schedule_type"),
      ])
)

rest_2 = (
    parking_meter_info_filtered.select(id_cols + ["restdates", "resttime2"])
      .rename({"restdates": "day_range", "resttime2": "time_range"})
      .with_columns([
          pl.lit("rest_time").alias("schedule_type"),
      ])
)

# concat different schedule_type
schedule_long = (
    pl.concat([active_1, active_2, rest_1, rest_2], how="vertical")
    .filter(pl.col("day_range").is_not_null() | pl.col("time_range").is_not_null())
)

schedule_long.sort(["node","location","x","y"]).head(30)

x,y,node,location,maxhours_hours,day_range,time_range,schedule_type
f64,f64,str,str,f64,str,str,str
-79.980303,40.428329,"""18-CARSO-L""","""18TH & CARSON LOT""",null,"""Fri - Sat""","""~ 8AM - 10PM""","""active_time"""
-79.980303,40.428329,"""18-CARSO-L""","""18TH & CARSON LOT""",null,"""Mon - Thurs""","""~ 8AM - 10PM""","""active_time"""
-79.980263,40.428655,"""18-CARSO-L""","""18TH & CARSON LOT""",null,"""Fri - Sat""","""~ 8AM - 10PM""","""active_time"""
-79.980263,40.428655,"""18-CARSO-L""","""18TH & CARSON LOT""",null,"""Mon - Thurs""","""~ 8AM - 10PM""","""active_time"""
-79.980585,40.42935,"""18-SIDNE-L""","""18TH & SIDNEY LOT""",null,"""Fri - Sat""","""~ 8AM - 10PM""","""active_time"""
-79.980585,40.42935,"""18-SIDNE-L""","""18TH & SIDNEY LOT""",null,"""Mon - Thurs""","""~ 8AM - 10PM""","""active_time"""
-79.980566,40.429427,"""18-SIDNE-L""","""18TH & SIDNEY LOT""",null,"""Fri - Sat""","""~ 8AM - 10PM""","""active_time"""
-79.980566,40.429427,"""18-SIDNE-L""","""18TH & SIDNEY LOT""",null,"""Mon - Thurs""","""~ 8AM - 10PM""","""active_time"""
-79.978399,40.428535,"""19-CARSO-L""","""19TH & CARSON LOT""",null,"""Fri - Sat""","""~ 8AM - 10PM""","""active_time"""


**Code that shows issue 2**

In [27]:
# Do rows having same node have all non-(x,y,location) columns identical?? 
cols_to_check = parking_meter_info_filtered.columns
cols_to_check = [c for c in cols_to_check if c not in ["x", "y", "node","location"]]

same_other_cols = (
    parking_meter_info_filtered.group_by("node")
      .agg([
          # create columns with either true or false value
          (pl.col(c).n_unique().fill_null(0) <= 1).alias(f"{c}_same")
          for c in cols_to_check
      ])
      .with_columns(
          # true if all columns are identical false other wise
          pl.all_horizontal(pl.exclude("node")).alias("all_other_cols_same")
      )
)

bad_locs = (
    same_other_cols
    .filter(~pl.col("all_other_cols_same"))
)

print(f"Number of node group that have different value in some columns: \
{bad_locs.height}")
bad_locs.head()

Number of node group that have different value in some columns: 62


node,date1_same,date2_same,time1_same,time2_same,restdates_same,resttime1_same,resttime2_same,rate_same,rate2_same,rate_hours_same,rate2_hours_same,ratetimea_same,ratetimeb_same,maxhours_hours_same,all_other_cols_same
str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
"""EASTCARS-L""",false,false,false,false,true,true,true,false,true,false,true,true,true,true,false
"""SHADYSIDE2""",false,true,false,true,false,false,true,false,true,false,true,true,true,false,false
"""BEECHVIEW""",false,true,false,true,true,true,true,false,true,false,true,true,true,false,false
"""SQ.HILL2""",false,true,false,true,true,true,true,false,true,false,true,true,true,false,false
"""NORTHSIDE""",false,true,false,true,false,false,true,false,true,false,true,true,true,false,false


#### 3.1 Broken Down into Two Dataframes

#### 3.1.1 Construct Parking Meters and Zones DF

**parking_meters**

In [ ]:
# drop rows with null value in these columns, bacause we are going to 
# build meter_id base on them
parking_meter_info_filtered = parking_meter_info_filtered.drop_nulls(["node","x","y"])

In [29]:
df = parking_meter_info_filtered

# choose rounding precision for key generation
ROUND_N = 5

parking_meter_info_meters_id = (
    df
    .with_columns([
        pl.col("x").round(ROUND_N,).alias("x_round"),
        pl.col("y").round(ROUND_N).alias("y_round"),
    ])
    # meter_id = a string key (zone + rounded coords)
    .with_columns(
        (pl.col("node") + "|" + pl.col("x_round").cast(pl.Utf8) + "|" + pl.col("y_round").cast(pl.Utf8))
        .alias("meter_id")
    )
)

parking_meters = (
    parking_meter_info_meters_id
    .select([
        pl.col("meter_id"),
        pl.col("node").alias("zone_id"),
        pl.col("location"),
        pl.col("x"),
        pl.col("y"),
        # pl.col("maxhours_hours"),
    ])
)
parking_meters.head()

meter_id,zone_id,location,x,y
str,str,str,f64,f64
"""SHER-HAR-L|-79.92283|40.46249""","""SHER-HAR-L""","""SHERIDAN HARVARD LOT""",-79.922827,40.46249
"""SHER-HAR-L|-79.92353|40.46272""","""SHER-HAR-L""","""SHERIDAN HARVARD LOT""",-79.92353,40.462718
"""SHER-KIR-L|-79.92313|40.4618""","""SHER-KIR-L""","""SHERIDAN KIRKWOOD LOT""",-79.923127,40.461796
"""SHER-KIR-L|-79.92269|40.46179""","""SHER-KIR-L""","""SHERIDAN KIRKWOOD LOT""",-79.922689,40.461792
"""SHER-KIR-L|-79.92233|40.46151""","""SHER-KIR-L""","""SHERIDAN KIRKWOOD LOT""",-79.922329,40.461512


In [30]:
# check wheter meter_id is unique
parking_meters.height == \
parking_meters.select("meter_id").n_unique()

True

**parking_zones**

In [31]:
parking_zones = (
    parking_meters
    .group_by("zone_id")
    .agg([
        # Adding location list
        pl.col("location")
          .drop_nulls()
          .unique()
          .sort()
          .alias("location_list"),
        # Adding number of locations
        pl.col("location")
          .drop_nulls()
          .n_unique()
          .alias("n_locations"),
        # Adding number of meters
        pl.len().alias("n_meters"),
        # # Optional: Adding min, max, mean coords
        # pl.col("x").min().alias("x_min"),
        # pl.col("x").max().alias("x_max"),
        # pl.col("y").min().alias("y_min"),
        # pl.col("y").max().alias("y_max"),
        # pl.col("x").mean().alias("x_centroid"),
        # pl.col("y").mean().alias("y_centroid"),
    ])
    .sort("zone_id")
    .with_columns(
        pl.col("location_list").list.join(", ").alias("location_list")
    )
)
parking_zones.head()

zone_id,location_list,n_locations,n_meters
str,str,u32,u32
"""18-CARSO-L""","""18TH & CARSON LOT""",1,2
"""18-SIDNE-L""","""18TH & SIDNEY LOT""",1,2
"""19-CARSO-L""","""19TH & CARSON LOT""",1,1
"""20-SIDNE-L""","""20TH & SIDNEY LOT""",1,2
"""42-BUTLE-L""","""42ND & BUTLER LOT""",1,1


In [32]:
# Virtual Terminal for ParkMobile doesn't seem like informative, let's drop it
parking_meters_physical = parking_meters.filter(
    ~pl.col("location").str.contains("Virtual", literal=False)
)

parking_zones = (
    parking_meters_physical
    .group_by("zone_id")
    .agg([
        # Adding location list
        pl.col("location")
          .drop_nulls()
          .unique()
          .sort()
          .alias("location_list"),
        # Adding number of locations
        pl.col("location")
          .drop_nulls()
          .n_unique()
          .alias("n_locations"),
        # Adding number of meters
        pl.len().alias("n_meters"),
    ])
    .sort("n_locations",descending=True)
    .with_columns(
        pl.col("location_list").list.join(", ").alias("location_list")
    )
)
parking_zones.head()

zone_id,location_list,n_locations,n_meters
str,str,u32,u32
"""NORTHSIDE""","""ALLEGHENY SQ, ARCH ST, BRIGHTO…",24,91
"""SOUTHSIDE""","""BEDFORD SQ, E CARSON ST, JANE …",23,101
"""DOWNTOWN1""","""10TH ST, 6TH AVE, 7TH AVE, 7TH…",18,41
"""DOWNTOWN2""","""1ST AVE, 2ND AVE, 3RD AVE, 4TH…",15,57
"""EASTLIB""","""BAUM BLVD, BROAD ST, CENTRE AV…",15,57


#### 3.1.2 Construct Parking Rates and Hours DF

**parking_rates**

In [33]:
parking_rates = (
    pl.concat([
        parking_meter_info_meters_id.select([
            pl.col("node").alias("zone_id"),
            pl.lit("A").alias("rate_window"),
            pl.col("ratetimea").alias("rate_time_raw"),
            pl.col("rate_hours").alias("rate_raw"),
            pl.col("meter_id")
        ]),
        parking_meter_info_meters_id.select([
            pl.col("node").alias("zone_id"),
            pl.lit("B").alias("rate_window"),
            pl.col("ratetimeb").alias("rate_time_raw"),
            pl.col("rate2_hours").alias("rate_raw"),
            pl.col("meter_id")
        ])
    ])
    # keep only real rate records
    .filter(pl.col("rate_raw").is_not_null())
    .with_columns([
        pl.col("rate_raw").alias("rate"),
        pl.col("rate_time_raw").alias("rate_time"),
    ])
    .select(["zone_id", "rate_window", "rate_time", "rate", "meter_id"])
    .sort(["zone_id", "rate_window"])
).with_columns(
    pl.col("rate_time").fill_null("all time")
)
parking_rates.head()

zone_id,rate_window,rate_time,rate,meter_id
str,str,str,f64,str
"""18-CARSO-L""","""A""","""all time""",1.5,"""18-CARSO-L|-79.9803|40.42833"""
"""18-CARSO-L""","""A""","""all time""",1.5,"""18-CARSO-L|-79.98026|40.42866"""
"""18-SIDNE-L""","""A""","""all time""",1.5,"""18-SIDNE-L|-79.98057|40.42943"""
"""18-SIDNE-L""","""A""","""all time""",1.5,"""18-SIDNE-L|-79.98058|40.42935"""
"""19-CARSO-L""","""A""","""all time""",1.5,"""19-CARSO-L|-79.9784|40.42853"""


**parking_hours**

In [34]:
def coalesce_range(a: pl.Expr, b: pl.Expr) -> pl.Expr:
    # If both exist and are different, join them; otherwise take whichever exists.
    # (Remember most of the time date2/time2 is null.)

    return (
        pl.when(a.is_not_null() & b.is_not_null() & (a != b))
          .then(a + " | " + b)
          .otherwise(pl.coalesce([a, b]))
    )

active_hours = (
    parking_meter_info_meters_id.select([
        pl.col("node").alias("zone_id"),
        pl.lit("active").alias("schedule_type"),
        coalesce_range(pl.col("date1"), pl.col("date2")).alias("day_range"),
        coalesce_range(pl.col("time1"), pl.col("time2")).alias("time_range"),
        pl.col("maxhours_hours"),
        pl.col("meter_id")
    ])
    .filter(pl.col("day_range").is_not_null() | pl.col("time_range").is_not_null())
)

rest_hours = (
    parking_meter_info_meters_id.select([
        pl.col("node").alias("zone_id"),
        pl.lit("rest").alias("schedule_type"),
        pl.col("restdates").alias("day_range"),
        coalesce_range(pl.col("resttime1"), pl.col("resttime2")).alias("time_range"),
        pl.col("maxhours_hours"),
        pl.col("meter_id")
    ])
    .filter(pl.col("day_range").is_not_null() | pl.col("time_range").is_not_null())
)

parking_hours = (
    pl.concat([active_hours, rest_hours])
    .select(["zone_id", "schedule_type", "day_range", "time_range", "maxhours_hours","meter_id"])
    .sort(["zone_id", "schedule_type"])
)

parking_hours.head()

zone_id,schedule_type,day_range,time_range,maxhours_hours,meter_id
str,str,str,str,f64,str
"""18-CARSO-L""","""active""","""Mon - Thurs | Fri - Sat""","""~ 8AM - 10PM""",null,"""18-CARSO-L|-79.9803|40.42833"""
"""18-CARSO-L""","""active""","""Mon - Thurs | Fri - Sat""","""~ 8AM - 10PM""",null,"""18-CARSO-L|-79.98026|40.42866"""
"""18-SIDNE-L""","""active""","""Mon - Thurs | Fri - Sat""","""~ 8AM - 10PM""",null,"""18-SIDNE-L|-79.98057|40.42943"""
"""18-SIDNE-L""","""active""","""Mon - Thurs | Fri - Sat""","""~ 8AM - 10PM""",null,"""18-SIDNE-L|-79.98058|40.42935"""
"""19-CARSO-L""","""active""","""Mon - Thurs | Fri - Sat""","""~ 8AM - 10PM""",null,"""19-CARSO-L|-79.9784|40.42853"""


**Save filtered dataset**

In [ ]:
# save brokendown datasets
parking_zones.write_csv("../cleaned_data/parking_meter_data/parking_zones.csv")
parking_meters.write_csv("../cleaned_data/parking_meter_data/parking_meters.csv")
parking_rates.write_csv("../cleaned_data/parking_meter_data/parking_rates.csv")
parking_hours.write_csv("../cleaned_data/parking_meter_data/parking_hours.csv")